<b><font color='aquamarine' size="6"> Data Preprocessing and Visualization</font></b>

<b><font size="6" color="aquamarine">City Hospital – Data Preprocessing &
Exploratory Analysis</font></b><a class="anchor"></a><a id="toc"></a><br>

### Group 16 - Team Members

| Name | Student Number |
|---|---|
| João Umbelino | 20251309 |
| Mariana Monteiri | 20251290 |
| Martim Violante | 20251310 |
| Rafael Portela | 20251361 |

---
## <font color='#BFD72F'>Table of Contents</font> <a class="anchor" id='toc'></a>
- [1. Part A - Data Understanding & Cleaning](#1)
  - [1.1 Loading and Importing the Data](#1_1)
    - [1.1.1 Initial Understanding of the Dataset](#1_1_2)
  - [1.2 Handling Duplicate Records](#1_2)
    - [1.2.1 Identifying Duplicate Rows](#1_2_1)
    - [1.2.2 Removing Duplicate Records](#1_2_2)
    - [1.2.3 Justifying the Cleaning Decisions](#1_2_3)
  - [1.3 Identifying and Handling Missing Values](#1_3)
    - [1.3.1 Identifying Missing Values Across Variables](#1_3_1)
    - [1.3.2 Analysing Missing Data Patterns](#1_3_2)
    - [1.3.3 Applying Imputation Strategies](#1_3_3)
  - [1.4 Handling Inconsistent Data](#1_4)
      - [1.4.1 Identifying Inconsistent Data](#1_4_1)
        - [1.4.1.1 Syntax, Data Types & Format Errors (Dates, Negative Values)](#1_4_1_1)
        - [1.4.1.2 Invalid Categorical Labels & Out-of-Scale Scores](#1_4_1_2)
        - [1.4.1.3 Demographics & Logical Contradictions (Insurance, Attributes)](#1_4_1_3)
        - [1.4.1.4 Cross-Visit & Longitudinal Inconsistencies](#1_4_1_4)
      - [1.4.2 Correcting and Standardizing Inconsistent Data](#1_4_2)
        - [1.4.2.1 Type Casting & Format Alignment](#1_4_2_1)
        - [1.4.2.2 Category Mapping & Text Standardization](#1_4_2_2)
        - [1.4.2.3 Conflict Resolution for Static & Longitudinal Patient Data](#1_4_2_3)
        - [1.4.2.4 Domain Rule Enforcement & Out-of-Bounds Remediation](#1_4_2_4)
  - [1.5 Identifying and Handling Outliers](#1_5)
    - [1.5.1 Detecting Outliers in Numerical Variables](#1_5_1)
    - [1.5.2 Visualizing Potential Outliers](#1_5_2)
    - [1.5.3 Deciding How to Handle Outliers](#1_5_3)

- [2. Part B - Feature Engineering & ABT Construction](#2)
  - [2.1 Patient-Level ABT Construction](#2_1)
    - [2.1.1 Static Attributes](#2_1_1)
    - [2.1.2 Visit Behaviour Features](#2_1_2)
    - [2.1.3 Financial Features](#2_1_3)
    - [2.1.4 Satisfaction Features](#2_1_4)
    - [2.1.5 Recency and Activity Features](#2_1_5)
  - [2.2 Data Dictionary](#2_2)
  - [2.3 Exporting the ABT](#2_3)

- [3. Part C - Exploratory Visualization & Insights](#3)
  - [3.1 Patient Demographics](#3_1)
    - [3.1.1 Age Distribution](#3_1_1)
    - [3.1.2 Gender Distribution](#3_1_2)
    - [3.1.3 Income Distribution](#3_1_3)
  - [3.2 Department and Service Usage](#3_2)
    - [3.2.1 Consultations by Department](#3_2_1)
    - [3.2.2 Consultations Over Time](#3_2_2)
    - [3.2.3 Department Usage by Gender or Age Group](#3_2_3)
  - [3.3 Patient Satisfaction](#3_3)
    - [3.3.1 Satisfaction by Department](#3_3_1)
    - [3.3.2 Satisfaction by Patient Segment](#3_3_2)
  - [3.4 Additional Visualizations](#3_4)
  - [3.5 Management Questions and Key Insights](#3_5)

---

### <font color='turquoise' size='6'>1. Part A – Data Understanding & Cleaning </font> <a class="anchor" id="1"></a>
  [Back to TOC](#toc)

In this first part, we perform a comprehensive diagnostic audit of the consultation dataset. We detect and resolve duplicate records, address missing values through domain rules and patient-level propagation, correct syntax, logical, and longitudinal inconsistencies, and identify numerical outliers through statistical fences and domain verification.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Image

warnings.filterwarnings("ignore")

# Configure publication-grade styling
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## <font color='turquoise'>1.1 Loading and Importing the Data</font> <a class="anchor" id="1_1"></a>
[Back to TOC](#toc)

We load the consultation records from `Project Data.xlsx`. The raw file is read once, and all downstream cleaning operations are applied on working copies to ensure 100% reproducibility.

In [ ]:
# Load raw consultation data
if os.path.exists("Project Data.xlsx"):
    filepath = "Project Data.xlsx"
elif os.path.exists("Project Data nova.xlsx"):
    filepath = "Project Data nova.xlsx"
else:
    filepath = "Project_Data.xlsx"

data = pd.read_excel(filepath)
df = data.copy()
print(f"Loaded '{filepath}' successfully: {df.shape[0]:,} rows and {df.shape[1]} columns.")

### <font color='turquoise'>1.1.1 Initial Understanding of the Dataset</font> <a class="anchor" id="1_1_2"></a>
[Back to TOC](#toc)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
print("Dataset dimensions :", df.shape)
print("Unique patient count:", df["Patient ID"].nunique())
print("Earliest visit date :", df["Visit Date"].min())
print("Latest visit date   :", df["Visit Date"].max())
print()
print("Data Types & Memory Usage:")
print(df.dtypes)

**First Impressions & Initial Assessment:**
- **Record Volume:** 10,008 consultation rows and 16 variables spanning 500 unique patients (`Patient ID`).
- **Observation Window:** 6 full months from **January 1, 2024 to June 30, 2024**, just prior to the specified reference date of July 1, 2024.
- **Variable Composition:**
  - *Demographic & Socio-Economic:* `Age`, `Gender`, `City of Residence`, `Profession`, `Education Level`, `Marital Status`, `Approximate Annual Income`.
  - *Clinical Context:* `Family History`.
  - *Administrative & Financial Terms:* `Insurance Provider`, `Insurance Coverage`, `Consultation Price`.
  - *Encounter Details:* `Visit Date`, `Department`, `Consultation Duration`, `Satisfaction Level`.

## <font color='turquoise'>1.2 Handling Duplicate Records</font> <a class="anchor" id="1_2"></a>
[Back to TOC](#toc)

### <font color='turquoise'>1.2.1 Identifying Duplicate Rows</font> <a class="anchor" id="1_2_1"></a>
[Back to TOC](#toc)

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Total exact duplicate rows detected: {n_duplicates}")
display(df[df.duplicated(keep=False)].sort_values(by=["Patient ID", "Visit Date"]))

### <font color='turquoise'>1.2.2 Removing Duplicate Records</font> <a class="anchor" id="1_2_2"></a>
[Back to TOC](#toc)

In [ ]:
# Dropping exact duplicate rows and resetting the index
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dataset shape after dropping exact duplicates: {df.shape}")
print(f"Unique patients retained: {df['Patient ID'].nunique()}")
print(f"Remaining duplicate rows: {df.duplicated().sum()}")

### <font color='turquoise'>1.2.3 Justifying the Cleaning Decisions</font> <a class="anchor" id="1_2_3"></a>
[Back to TOC](#toc)

| Finding | Remediation Decision | Technical & Domain Justification |
|---|---|---|
| **8 exact duplicate rows** | **Remove (keep first occurrence)** | The 8 records are identical across all 16 attributes, down to the exact timestamp and high-precision consultation price (up to 15 decimal places). These represent accidental double-ingestion of the exact same consultation event. Dropping them removes redundant double-counting without altering the total cohort of 500 patients. |

## <font color='turquoise'>1.3 Identifying and Handling Missing Values</font> <a class="anchor" id="1_3"></a>
[Back to TOC](#toc)

### <font color='turquoise'>1.3.1 Identifying Missing Values Across Variables</font> <a class="anchor" id="1_3_1"></a>
[Back to TOC](#toc)

In [ ]:
missing_summary = pd.DataFrame({
    "Non-Null Count": df.notnull().sum(),
    "Missing Count": df.isnull().sum(),
    "Missing (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
display(missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing Count", ascending=False))

print("Total visits with at least one missing value:", df.isna().any(axis=1).sum())
print("Total visits with more than one missing value:", (df.isna().sum(axis=1) > 1).sum())

# Check for hidden placeholder strings (e.g. '?', 'N/A', empty strings) in text columns
hidden_candidates = ["", " ", "?", "n/a", "na", "nan", "null", "unknown", "-"]
text_cols = df.select_dtypes(include="object").columns
hidden_counts = {col: df[col].astype(str).str.strip().str.lower().isin(hidden_candidates).sum() for col in text_cols}
print("Hidden placeholder counts:", {k: v for k, v in hidden_counts.items() if v > 0})

**Detailed Audit of Missingness:**
- **5 variables** present genuine missing values: `Age` (56 rows, 0.56%), `Education Level` (29 rows, 0.29%), `Insurance Provider` (104 rows, 1.04%), `Insurance Coverage` (49 rows, 0.49%), and `Approximate Annual Income` (154 rows, 1.54%).
- **Structural Nulls in `Family History` & `Insurance Provider`:** Standard Excel reading converts the text string `"None"` to `NaN`. In this dataset, `"None"` in `Family History` indicates that the patient has no reported family history of hereditary illness; `"None"` in `Insurance Provider` identifies uninsured / self-paying patients who legitimately have £0.00 coverage. Only the 104 blank entries in Insurance Provider represent true administrative omissions.
- Overall missingness is very low (< 1.6% per variable), and almost never co-occurs on the same row.

### <font color='turquoise'>1.3.2 Analysing Missing Data Patterns</font> <a class="anchor" id="1_3_2"></a>
[Back to TOC](#toc)

In [ ]:
cols_with_missing = ["Age", "Education Level", "Insurance Provider", "Insurance Coverage", "Approximate Annual Income"]

print("--- Patient-Level Redundancy Check ---")
for col in cols_with_missing:
    pts_affected = df.loc[df[col].isna(), "Patient ID"].nunique()
    pts_all_missing = (df.groupby("Patient ID")[col].count() == 0).sum()
    print(f"{col:28s} | Affected Patients: {pts_affected:3d} | Patients with NO value across all visits: {pts_all_missing}")

print("
--- Percentage of Missing Values by Department (%) ---")
display((df[cols_with_missing].isna().groupby(df["Department"]).mean() * 100).round(1))

**Pattern Insights:**
- **Complete Patient Information in Other Encounters:** For `Age`, `Education Level`, and `Approximate Annual Income`, **100% of affected patients have complete, valid records in their other consultations**. The only exception is `Insurance Provider`, where exactly 2 patients have no provider specified on any visit.
- **Departmental Capture Artifact (MCAR / MAR):** `Age` is missing exclusively across 5 departments; `Approximate Annual Income` is missing exclusively across 6 other departments. This proves that missingness stems from localized departmental data-entry workflows rather than patient self-selection or sensitive attributes.
- **Static Feature Invariance:** Because personal attributes do not fluctuate over a 6-month period, other visits from the same patient provide exact ground truth.

### <font color='turquoise'>1.3.3 Applying Imputation Strategies</font> <a class="anchor" id="1_3_3"></a>
[Back to TOC](#toc)

| Variable | Missing Gaps | Imputation Strategy | Technical & Domain Justification |
|---|---|---|---|
| **Age** | 56 visits | Patient-level forward/backward fill | Static attribute per patient; other visits provide the exact age with certainty. |
| **Education Level** | 29 visits | Patient-level forward/backward fill | Static educational attainment; other visits provide the true qualification. |
| **Insurance Provider** | 104 visits | Fill from patient's other visits; infer from `Coverage / Price` | 50 rows filled from patient's own visits. For the 2 patients with no provider on any visit (Patients 1076, 1405), `Coverage / Price` ratio (0.70 and 0.90) uniquely identifies Provider C and Provider B. |
| **Insurance Coverage** | 49 visits | Deterministic formula: `Provider Rate * Price` | Coverage is contractual (Provider A: 80%, B: 90%, C: 70%, D: 100%, Uninsured: 0%). |
| **Approximate Annual Income** | 154 visits | Patient median valid income | Robust against minor visit-to-visit variations and transcription noise. |
| **Family History** | 1,990 rows | Recode structural blanks to `"None"` | Represents absence of family medical condition, not lost data. |

In [ ]:
# 1. Age and Education Level: Within-patient propagation
for col in ["Age", "Education Level"]:
    df[col] = df[col].fillna(df.groupby("Patient ID")[col].transform("first"))

# 2. Insurance Provider: Propagate within patient
df["Insurance Provider"] = df["Insurance Provider"].fillna(df.groupby("Patient ID")["Insurance Provider"].transform("first"))

# For patients with no provider on any visit, deduce from Coverage / Price ratio
ratio = df["Insurance Coverage"] / df["Consultation Price"]
provider_by_rate = {0.0: "Uninsured", 0.7: "Provider C", 0.8: "Provider A", 0.9: "Provider B", 1.0: "Provider D"}
patient_ratio = ratio.groupby(df["Patient ID"]).transform("median").round(1)
df["Insurance Provider"] = df["Insurance Provider"].fillna(patient_ratio.map(provider_by_rate)).fillna("Uninsured")

# 3. Insurance Coverage: Recompute using contractual provider rates
rate_by_provider = {"Uninsured": 0.0, "None": 0.0, "Provider A": 0.8, "Provider B": 0.9, "Provider C": 0.7, "Provider D": 1.0}
rate_series = df["Insurance Provider"].map(rate_by_provider).fillna(0.0)
df["Insurance Coverage"] = df["Insurance Coverage"].fillna((rate_series * df["Consultation Price"]).round(2))

# 4. Approximate Annual Income: Impute with patient's median valid income
df["Approximate Annual Income"] = df["Approximate Annual Income"].fillna(
    df.groupby("Patient ID")["Approximate Annual Income"].transform("median")
)

# 5. Family History: Standardize blanks to "None"
df["Family History"] = df["Family History"].fillna("None")

print("Missing values after applying imputation strategies:")
display(df[cols_with_missing + ["Family History"]].isnull().sum().to_frame("Remaining Missing"))

## <font color='turquoise'>1.4 Handling Inconsistent Data</font> <a class="anchor" id="1_4"></a>
[Back to TOC](#toc)

In this section, we systematically audit data types, syntax errors, out-of-scale survey responses, logical cross-variable contradictions, and longitudinal inconsistencies across patient visits.

### <font color='turquoise'>1.4.1 Identifying Inconsistent Data</font> <a class="anchor" id="1_4_1"></a>
[Back to TOC](#toc)

#### <font color='turquoise'>1.4.1.1 Syntax, Data Types & Format Errors (Dates, Negative Values)</font> <a class="anchor" id="1_4_1_1"></a>
[Back to TOC](#toc)

In [ ]:
num_cols = ["Age", "Consultation Duration", "Satisfaction Level", "Approximate Annual Income", "Consultation Price", "Insurance Coverage"]

# 1. Negative values audit
negative_counts = (df[num_cols] < 0).sum()
print("Negative values count across numerical variables:")
print(negative_counts)

# 2. Date boundary audit
min_date = df["Visit Date"].min()
max_date = df["Visit Date"].max()
invalid_dates = ((df["Visit Date"] < "2024-01-01") | (df["Visit Date"] > "2024-06-30")).sum()
print(f"Earliest consultation date: {min_date}")
print(f"Latest consultation date  : {max_date}")
print(f"Dates outside 6-month observation window: {invalid_dates}")

**Findings (Syntax & Date Formats):**
- **Negative Values:** Exactly 0 negative values across all numerical variables.
- **Date Validity:** All 10,000 visits fall strictly between January 1, 2024 and June 30, 2024. No corrupted or out-of-bounds timestamps exist.

#### <font color='turquoise'>1.4.1.2 Invalid Categorical Labels & Out-of-Scale Scores</font> <a class="anchor" id="1_4_1_2"></a>
[Back to TOC](#toc)

In [ ]:
cat_cols = ["Gender", "City of Residence", "Profession", "Department", "Education Level", "Marital Status", "Insurance Provider"]
for col in cat_cols:
    print(f"{col:20s}: {sorted(df[col].dropna().unique())}")

print("
--- Satisfaction Level Value Distribution ---")
display(df["Satisfaction Level"].value_counts().sort_index().to_frame("Count"))
sat_6_rows = (df["Satisfaction Level"] == 6).sum()
print(f"
Total visits with Satisfaction Level = 6: {sat_6_rows}")
print("Department breakdown for Satisfaction Level = 6:")
display(df.loc[df["Satisfaction Level"] == 6, "Department"].value_counts().to_frame("Count (Rating=6)"))

**Findings (Labels & Satisfaction Scale):**
- All categorical labels are clean with standard casing and no leading or trailing whitespace.
- **Out-of-Scale Satisfaction Score:** `Satisfaction Level` is defined on a **1 to 5 Likert scale**, yet **719 records contain a score of 6**. Crucially, **99% of these 6s originate from just two departments**: **ENT** (368 records, 36.3% of its ratings) and **General Practice** (344 records, 36.6% of its ratings). This reveals a systematic survey configuration error in those two clinical intake forms.

#### <font color='turquoise'>1.4.1.3 Demographics & Logical Contradictions (Insurance, Attributes)</font> <a class="anchor" id="1_4_1_3"></a>
[Back to TOC](#toc)

In [ ]:
# 1. Pediatric Contradictions: Age < 18 with adult professions or marital status
pediatric = df[df["Age"] < 18]
ped_adult_prof = pediatric[pediatric["Profession"].isin(["Doctor", "Lawyer", "Engineer", "Teacher", "Retired", "Artist"])]
ped_married = pediatric[pediatric["Marital Status"].isin(["Married", "Divorced", "Widowed"])]

print(f"Pediatric patients (< 18 yrs) with adult professions: {ped_adult_prof['Patient ID'].nunique()} patients ({len(ped_adult_prof)} visits)")
display(ped_adult_prof[["Patient ID", "Age", "Profession", "Education Level", "Approximate Annual Income"]].drop_duplicates())

# 2. Financial Contradictions: Insurance Coverage > Consultation Price
cov_exceeds_price = df[df["Insurance Coverage"] > df["Consultation Price"]]
print(f"
Visits where Insurance Coverage exceeds Consultation Price: {len(cov_exceeds_price)}")
display(cov_exceeds_price[["Patient ID", "Department", "Insurance Provider", "Consultation Price", "Insurance Coverage"]])

**Findings (Logical Contradictions):**
- **Pediatric Contradictions:** 6 children aged 1 to 15 are labeled with adult professions (e.g. Lawyer, Doctor, Teacher) or marital statuses (Married, Divorced), despite having school-level education and £0 income. These represent transcription errors.
- **Financial Contradictions:** 6 visits record an `Insurance Coverage` greater than the `Consultation Price` (e.g., £291.60 coverage on a £243.00 price), which violates basic insurance contract rules.

#### <font color='turquoise'>1.4.1.4 Cross-Visit & Longitudinal Inconsistencies</font> <a class="anchor" id="1_4_1_4"></a>
[Back to TOC](#toc)

In [ ]:
static_attributes = ["Gender", "City of Residence", "Profession", "Marital Status", "Education Level", "Family History", "Insurance Provider"]
inconsistent_counts = {}
for col in static_attributes:
    n_inconsistent = (df.groupby("Patient ID")[col].nunique() > 1).sum()
    inconsistent_counts[col] = n_inconsistent
    print(f"{col:20s} | Patients with conflicting values across visits: {n_inconsistent}")

**Findings (Longitudinal Conflicts):**
- In a 6-month observation window, core demographic and biographical attributes should not change.
- We observe conflicting entries for 5 patients in `Gender` (e.g., 19 visits as Male and 1 visit as Female) and 4 patients in `Marital Status`, alongside occasional variations in `Profession`. These reflect clerical noise during front-desk registration.

### <font color='turquoise'>1.4.2 Correcting and Standardizing Inconsistent Data</font> <a class="anchor" id="1_4_2"></a>
[Back to TOC](#toc)

#### <font color='turquoise'>1.4.2.1 Type Casting & Format Alignment</font> <a class="anchor" id="1_4_2_1"></a>
[Back to TOC](#toc)

In [ ]:
df["Visit Date"] = pd.to_datetime(df["Visit Date"])
df["Patient ID"] = df["Patient ID"].astype(int)
print("Standardized schema data types:")
print(df[["Patient ID", "Visit Date", "Consultation Duration", "Consultation Price", "Insurance Coverage"]].dtypes)

#### <font color='turquoise'>1.4.2.2 Category Mapping & Text Standardization</font> <a class="anchor" id="1_4_2_2"></a>
[Back to TOC](#toc)

In [ ]:
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

print("All categorical string fields stripped and standardized.")

#### <font color='turquoise'>1.4.2.3 Conflict Resolution for Static & Longitudinal Patient Data</font> <a class="anchor" id="1_4_2_3"></a>
[Back to TOC](#toc)

In [ ]:
# Standardize static attributes to the patient's most frequent (modal) valid value
for col in static_attributes:
    df[col] = df.groupby("Patient ID")[col].transform(lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else s.iloc[0])

# Enforce pediatric domain rules for Age < 18
child_mask = df["Age"] < 18
df.loc[child_mask, "Profession"] = "Student"
df.loc[child_mask, "Marital Status"] = "Single"
df.loc[child_mask, "Approximate Annual Income"] = 0.0

# Verify zero longitudinal conflicts remain
total_remaining_conflicts = sum((df.groupby("Patient ID")[col].nunique() > 1).sum() for col in static_attributes)
print(f"Total static attribute conflicts remaining across all patients: {total_remaining_conflicts}")

#### <font color='turquoise'>1.4.2.4 Domain Rule Enforcement & Out-of-Bounds Remediation</font> <a class="anchor" id="1_4_2_4"></a>
[Back to TOC](#toc)

In [ ]:
# 1. Satisfaction Level: Set invalid score of 6 to NaN
sat_6_remediated = int((df["Satisfaction Level"] == 6).sum())
df.loc[df["Satisfaction Level"] == 6, "Satisfaction Level"] = np.nan
print(f"Remediated {sat_6_remediated} invalid satisfaction scores (value=6) to NaN.")

# 2. Insurance Coverage & Out-of-Pocket Cost: Enforce contractual rate and ensure Coverage <= Price
rate_series = df["Insurance Provider"].map(rate_by_provider).fillna(0.0)
df["Insurance Coverage"] = np.minimum((rate_series * df["Consultation Price"]).round(2), df["Consultation Price"])
df["Out_of_Pocket"] = (df["Consultation Price"] - df["Insurance Coverage"]).round(2)

print(f"Enforced financial bounds: 0 rows have Coverage > Price or Out_of_Pocket < 0.")

**Remediation Justifications:**
- **Satisfaction Score (Value = 6):** Capping score 6 to 5 would artificially inflate ENT and General Practice into the top-rated departments in the hospital. Setting them to missing (`NaN`) prevents distortion of the genuine 1–5 scale while preserving the encounter records.
- **Financial Bounds:** Recomputing `Insurance Coverage` ensures strict compliance with provider reimbursement schedules and guarantees that patient out-of-pocket expenses are non-negative.

## <font color='turquoise'>1.5 Identifying and Handling Outliers</font> <a class="anchor" id="1_5"></a>
[Back to TOC](#toc)

We evaluate numerical variables for extreme anomalies using statistical distribution fences (1.5 * IQR) and domain logic.

### <font color='turquoise'>1.5.1 Detecting Outliers in Numerical Variables</font> <a class="anchor" id="1_5_1"></a>
[Back to TOC](#toc)

In [ ]:
num_vars = ["Age", "Consultation Duration", "Satisfaction Level", "Approximate Annual Income", "Consultation Price", "Insurance Coverage"]
display(df[num_vars].describe().round(2))

# 1.5 * IQR Rule Assessment
iqr_summary = []
for col in num_vars:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    n_low = (s < lower_fence).sum()
    n_high = (s > upper_fence).sum()
    iqr_summary.append({
        "Variable": col, "Q1": round(q1, 1), "Q3": round(q3, 1), "IQR": round(iqr, 1),
        "Lower Fence": round(lower_fence, 1), "Upper Fence": round(upper_fence, 1),
        "Outliers (Low)": n_low, "Outliers (High)": n_high, "Total IQR Outliers": n_low + n_high
    })
display(pd.DataFrame(iqr_summary))

**Critique of Naive IQR Outlier Detection:**
- **Consultation Duration:** The IQR upper fence (173.5 min) cleanly flags the 10 extreme durations (228 to 600 min), while legitimate consultations strictly follow standard 15–120 minute blocks.
- **Consultation Price:** Pooled IQR flags 241 consultations because prices legitimately vary by clinical specialty (£50–£150 in General Practice vs £250–£400 in Cardiology/Emergency). However, 7 consultations exhibit extreme typos of £25,300 to £39,999.
- **Approximate Annual Income:** Pooled IQR flags 2,285 rows due to the heavy density of legitimate £0 earners (students and children). Among adult earners, 21 records show extreme decimal-shift values from £1.1M up to £11.97M.
- **Age:** 103 records across 5 patients have ages between 118 and 195 due to an extra hundreds digit '1'.

### <font color='turquoise'>1.5.2 Visualizing Potential Outliers</font> <a class="anchor" id="1_5_2"></a>
[Back to TOC](#toc)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
plot_vars = [
    ("Age", axes[0], "Age (Years)", "Age"),
    ("Consultation Duration", axes[1], "Duration (Minutes)", "Consultation Duration"),
    ("Consultation Price", axes[2], "Price (£)", "Consultation Price"),
    ("Approximate Annual Income", axes[3], "Income (£)", "Approximate Annual Income")
]

for var, ax, ylabel, title in plot_vars:
    sns.boxplot(data=df, y=var, ax=ax, color="#4c72b0")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)

plt.tight_layout()
plt.show()

**Visual Diagnosis:**
- Each variable features a dense, well-bounded bulk of plausible clinical observations accompanied by isolated points orders of magnitude higher.
- These extreme values are unmistakable data-entry errors (extra zeros, scale shifts) rather than natural statistical tails.

### <font color='turquoise'>1.5.3 Deciding How to Handle Outliers</font> <a class="anchor" id="1_5_3"></a>
[Back to TOC](#toc)

| Variable | Outlier Observations | Remediation Action | Technical & Domain Justification |
|---|---|---|---|
| **Age** | 103 rows (5 patients) > 110 (118, 138, 142, 158, 195) | Subtract 100 | Clear extra leading digit '1'; Patient 1198 had 38 on one visit and 138 on others. Corrected ages (18, 38, 42, 58, 95) align with education level and profession. |
| **Consultation Price** | 7 rows > £1,000 (£25k-£40k) | Divide by 100 | Two-digit decimal shift typo; results (£253-£399) fall squarely within the corresponding departments' normal price bands. |
| **Approximate Annual Income** | 21 rows >= £1,000,000 (£1.1M-£11.97M) | Divide by 10 or 100 / patient median | Decimal shift correction brings values to £113k-£120k, aligning with upper bounds of valid adult earners and patients' other visits. |
| **Consultation Duration** | 10 rows > 120 min (228-600 min) | Impute department median | Valid consultation schedules never exceed 120 minutes; department median provides realistic clinical duration without discarding rows. |
| **Annual Income = £0** | 110 patients | Retain without alteration | Legitimate economic status for students and pediatric patients. |

In [ ]:
# 1. Correct Age typos (> 110: subtract 100)
df.loc[df["Age"] > 110, "Age"] = df.loc[df["Age"] > 110, "Age"] - 100
df["Age"] = df.groupby("Patient ID")["Age"].transform("first").astype(int)

# 2. Correct Consultation Price typos (> 1000: divide by 100)
df.loc[df["Consultation Price"] > 1000, "Consultation Price"] /= 100

# 3. Correct Annual Income typos (>= 10M: /100; >= 1M: /10)
df.loc[df["Approximate Annual Income"] >= 10_000_000, "Approximate Annual Income"] /= 100
df.loc[df["Approximate Annual Income"] >= 1_000_000, "Approximate Annual Income"] /= 10
# Impute with patient's median valid income
df["Approximate Annual Income"] = df.groupby("Patient ID")["Approximate Annual Income"].transform("median").round(2)
df.loc[df["Age"] < 18, "Approximate Annual Income"] = 0.0

# 4. Consultation Duration: Impute extreme values (> 120 min) with department median
dept_dur_median = df[df["Consultation Duration"] <= 120].groupby("Department")["Consultation Duration"].median()
df.loc[df["Consultation Duration"] > 120, "Consultation Duration"] = df["Department"].map(dept_dur_median)

# 5. Re-synchronize Insurance Coverage and Out_of_Pocket
rate_series = df["Insurance Provider"].map(rate_by_provider).fillna(0.0)
df["Insurance Coverage"] = (rate_series * df["Consultation Price"]).round(2)
df["Out_of_Pocket"] = (df["Consultation Price"] - df["Insurance Coverage"]).round(2)

# Post-treatment visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for var, ax, ylabel, title in plot_vars:
    sns.boxplot(data=df, y=var, ax=ax, color="#55a868")
    ax.set_title(f"{title} (Cleaned)", fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)

plt.tight_layout()
plt.show()

print("Cleaned Numerical Variables Summary:")
display(df[num_vars].describe().round(2))

**Summary of Part A Cleaning Decisions:**
- **Duplicates:** Removed 8 redundant rows (10,008 → 10,000 consultations); retained all 500 patients.
- **Missing Values:** Imputed static features via within-patient propagation; derived missing providers via Coverage/Price ratio; calculated coverage via provider contractual rates.
- **Inconsistencies:** Resolved cross-visit static variations to patient modal profile; enforced pediatric rules (<18 → Student, Single, £0); capped invalid coverage.
- **Outliers:** Subtracted 100 from age typos; scaled back decimal-shifted prices and incomes; imputed department medians for extreme durations; set invalid score 6 to missing.
- **Integrity:** The cleaned dataset has exactly 10,000 visits, 500 unique patients, and zero unintended missing values.

# <font color='turquoise' size='6'>2. Part B - Feature Engineering & ABT Construction</font> <a class="anchor" id="2"></a>
[Back to TOC](#toc)

In Part B, we aggregate the cleaned visit-level records into an **Analytic Base Table (ABT)** at the patient level (exactly 500 rows, 1 row per unique patient). All recency calculations use the fixed project reference date of **July 1, 2024**.

## <font color='turquoise'>2.1 Patient-Level ABT Construction</font> <a class="anchor" id="2_1"></a>
[Back to TOC](#toc)

### <font color='turquoise'>2.1.1 Static Attributes</font> <a class="anchor" id="2_1_1"></a>
[Back to TOC](#toc)

In [ ]:
ref_date = pd.Timestamp("2024-07-01")

# Helper visit-level indicator columns
df["Is_Weekend"] = df["Visit Date"].dt.dayofweek.isin([5, 6]).astype(int)
df["Is_Weekday"] = (~df["Visit Date"].dt.dayofweek.isin([5, 6])).astype(int)
df["Is_Low_Satisfaction"] = (df["Satisfaction Level"] <= 2).astype(int)

grouped = df.groupby("Patient ID")

# 1. Static Demographic & Socio-Economic Features
static_agg = grouped.agg(
    Age=("Age", "first"),
    Gender=("Gender", "first"),
    City_of_Residence=("City of Residence", "first"),
    Profession=("Profession", "first"),
    Insurance_Provider=("Insurance Provider", "first"),
    Family_History=("Family History", "first"),
    Education_Level=("Education Level", "first"),
    Marital_Status=("Marital Status", "first"),
    Approx_Annual_Income=("Approximate Annual Income", "median")
).reset_index()

# Derived Segmentation Features: 10-Year Age Bins & Income Quartiles
age_bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 105]
age_labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80-89", "90+"]
static_agg["Age_Group_10yr"] = pd.cut(
    static_agg["Age"], bins=age_bins, labels=age_labels, right=False, include_lowest=True
).astype(str)

static_agg["Income_Quartile"] = pd.qcut(
    static_agg["Approx_Annual_Income"].rank(method="first"),
    q=4,
    labels=["Q1 (Lowest)", "Q2 (Lower-Mid)", "Q3 (Upper-Mid)", "Q4 (Highest)"]
).astype(str)

display(static_agg.head())

### <font color='turquoise'>2.1.2 Visit Behaviour Features</font> <a class="anchor" id="2_1_2"></a>
[Back to TOC](#toc)

In [ ]:
def get_mode_dept(series):
    return series.value_counts().index[0]

# 2. Visit Behaviour Features
behaviour_agg = grouped.agg(
    Total_Visits=("Visit Date", "count"),
    Distinct_Departments=("Department", "nunique"),
    Most_Frequent_Department=("Department", get_mode_dept),
    Avg_Consultation_Duration=("Consultation Duration", "mean"),
    Total_Consultation_Duration=("Consultation Duration", "sum"),
    Weekend_Visits=("Is_Weekend", "sum"),
    Weekday_Visits=("Is_Weekday", "sum")
).reset_index()

behaviour_agg["Weekend_to_Weekday_Ratio"] = (
    behaviour_agg["Weekend_Visits"] / behaviour_agg["Weekday_Visits"].replace(0, 1)
).round(4)
behaviour_agg["Weekend_Visit_Share"] = (
    behaviour_agg["Weekend_Visits"] / behaviour_agg["Total_Visits"]
).round(4)

display(behaviour_agg.head())

### <font color='turquoise'>2.1.3 Financial Features</font> <a class="anchor" id="2_1_3"></a>
[Back to TOC](#toc)

In [ ]:
# 3. Financial Features
financial_agg = grouped.agg(
    Total_Consultation_Price=("Consultation Price", "sum"),
    Avg_Consultation_Price=("Consultation Price", "mean"),
    Total_Insurance_Coverage=("Insurance Coverage", "sum"),
    Avg_Insurance_Coverage=("Insurance Coverage", "mean"),
    Total_Out_of_Pocket=("Out_of_Pocket", "sum"),
    Avg_Out_of_Pocket=("Out_of_Pocket", "mean")
).reset_index()

financial_agg["Out_of_Pocket_Ratio"] = (
    financial_agg["Total_Out_of_Pocket"] / financial_agg["Total_Consultation_Price"]
).round(4)

display(financial_agg.head())

### <font color='turquoise'>2.1.4 Satisfaction Features</font> <a class="anchor" id="2_1_4"></a>
[Back to TOC](#toc)

In [ ]:
# 4. Satisfaction Features (Using valid 1-5 scores)
satisfaction_agg = grouped.agg(
    Avg_Satisfaction_Rating=("Satisfaction Level", "mean"),
    Min_Satisfaction_Rating=("Satisfaction Level", "min"),
    Max_Satisfaction_Rating=("Satisfaction Level", "max"),
    Low_Rating_Count=("Is_Low_Satisfaction", "sum")
).reset_index()

satisfaction_agg["Share_Low_Ratings"] = (
    satisfaction_agg["Low_Rating_Count"] / behaviour_agg["Total_Visits"]
).round(4)

display(satisfaction_agg.head())

### <font color='turquoise'>2.1.5 Recency and Activity Features</font> <a class="anchor" id="2_1_5"></a>
[Back to TOC](#toc)

In [ ]:
# 5. Recency & Activity Features (Relative to 2024-07-01)
recency_agg = grouped.agg(
    First_Visit_Date=("Visit Date", "min"),
    Last_Visit_Date=("Visit Date", "max")
).reset_index()

recency_agg["Days_Since_First_Visit"] = (ref_date - recency_agg["First_Visit_Date"]).dt.days
recency_agg["Days_Since_Last_Visit"] = (ref_date - recency_agg["Last_Visit_Date"]).dt.days
recency_agg["Active_Span_Days"] = (
    recency_agg["Days_Since_First_Visit"] - recency_agg["Days_Since_Last_Visit"]
).clip(lower=1)
recency_agg["Visits_Per_Month"] = (behaviour_agg["Total_Visits"] / 6.0).round(2)

# Merge all feature blocks into a single comprehensive ABT
abt = (
    static_agg
    .merge(behaviour_agg, on="Patient ID")
    .merge(financial_agg, on="Patient ID")
    .merge(satisfaction_agg, on="Patient ID")
    .merge(recency_agg.drop(columns=["First_Visit_Date", "Last_Visit_Date"]), on="Patient ID")
)

# Standardize floating-point rounding
float_cols_2dec = [
    "Approx_Annual_Income", "Avg_Consultation_Duration", "Total_Consultation_Price",
    "Avg_Consultation_Price", "Total_Insurance_Coverage", "Avg_Insurance_Coverage",
    "Total_Out_of_Pocket", "Avg_Out_of_Pocket", "Avg_Satisfaction_Rating"
]
for col in float_cols_2dec:
    abt[col] = abt[col].round(2)

print(f"ABT construction complete: exactly {abt.shape[0]} rows x {abt.shape[1]} features.")

## <font color='turquoise'>2.2 Data Dictionary</font> <a class="anchor" id="2_2"></a>
[Back to TOC](#toc)

In [ ]:
data_dict_records = [
    ("Patient ID", "Identifier", "Unique numerical patient identifier (Primary Key, N=500).", "int64"),
    ("Age", "Static Demographic", "Patient age in years (validated and standardized).", "int64"),
    ("Gender", "Static Demographic", "Patient gender (Female, Male, Other).", "str"),
    ("City_of_Residence", "Static Demographic", "City of residence across 8 UK/Irish metropolitan areas.", "str"),
    ("Profession", "Static Socio-Economic", "Patient occupation (standardized; <18 classified as Student).", "str"),
    ("Insurance_Provider", "Static Financial", "Health insurance provider (Provider A-D or 'Uninsured').", "str"),
    ("Family_History", "Static Clinical", "Recorded hereditary condition (Cancer, Diabetes, Heart Disease, Hypertension, None).", "str"),
    ("Education_Level", "Static Socio-Economic", "Highest educational attainment level (Pre-school to PhD).", "str"),
    ("Marital_Status", "Static Demographic", "Marital status (Single, Married, Divorced, Widowed).", "str"),
    ("Approx_Annual_Income", "Static Socio-Economic", "Median cleaned annual income (£) of the patient.", "float64"),
    ("Age_Group_10yr", "Static Derived", "10-year age bracket (0-9, 10-19, ..., 90+) for demographic grouping.", "category"),
    ("Income_Quartile", "Static Derived", "Income quartile segment (Q1 Lowest to Q4 Highest).", "category"),
    ("Total_Visits", "Visit Behaviour", "Total number of consultations between Jan 1 and Jun 30, 2024.", "int64"),
    ("Distinct_Departments", "Visit Behaviour", "Count of unique clinical departments visited by the patient.", "int64"),
    ("Most_Frequent_Department", "Visit Behaviour", "Modal (most frequently visited) hospital department.", "str"),
    ("Avg_Consultation_Duration", "Visit Behaviour", "Mean consultation duration in minutes across all visits.", "float64"),
    ("Total_Consultation_Duration", "Visit Behaviour", "Cumulative duration in minutes spent across all consultations.", "float64"),
    ("Weekend_Visits", "Visit Behaviour", "Count of consultations taking place on Saturday or Sunday.", "int64"),
    ("Weekday_Visits", "Visit Behaviour", "Count of consultations taking place Monday through Friday.", "int64"),
    ("Weekend_to_Weekday_Ratio", "Visit Behaviour", "Ratio of weekend visits to weekday visits (Weekend / max(1, Weekday)).", "float64"),
    ("Weekend_Visit_Share", "Visit Behaviour", "Proportion of patient visits occurring on weekends.", "float64"),
    ("Total_Consultation_Price", "Financial", "Cumulative gross billed consultation cost (£) across all visits.", "float64"),
    ("Avg_Consultation_Price", "Financial", "Mean gross price (£) per consultation.", "float64"),
    ("Total_Insurance_Coverage", "Financial", "Total amount (£) paid by insurance provider across all visits.", "float64"),
    ("Avg_Insurance_Coverage", "Financial", "Mean insurance coverage (£) per consultation.", "float64"),
    ("Total_Out_of_Pocket", "Financial", "Total out-of-pocket expenditure (£) paid by patient (Price - Coverage).", "float64"),
    ("Avg_Out_of_Pocket", "Financial", "Mean out-of-pocket expenditure (£) per consultation.", "float64"),
    ("Out_of_Pocket_Ratio", "Financial", "Proportion of total consultation costs paid out-of-pocket by patient.", "float64"),
    ("Avg_Satisfaction_Rating", "Satisfaction", "Mean satisfaction rating (1 = lowest, 5 = highest; valid ratings only).", "float64"),
    ("Min_Satisfaction_Rating", "Satisfaction", "Lowest satisfaction rating submitted by patient across visits.", "float64"),
    ("Max_Satisfaction_Rating", "Satisfaction", "Highest satisfaction rating submitted by patient across visits.", "float64"),
    ("Low_Rating_Count", "Satisfaction", "Number of consultations receiving a dissatisfied score (<= 2).", "int64"),
    ("Share_Low_Ratings", "Satisfaction", "Proportion of patient consultations rated <= 2.", "float64"),
    ("Days_Since_First_Visit", "Recency & Activity", "Days elapsed from patient's first consultation to July 1, 2024.", "int64"),
    ("Days_Since_Last_Visit", "Recency & Activity", "Days elapsed from patient's most recent consultation to July 1, 2024.", "int64"),
    ("Active_Span_Days", "Recency & Activity", "Days between first and last recorded visit (clinical engagement span).", "int64"),
    ("Visits_Per_Month", "Recency & Activity", "Average monthly consultation frequency (Total_Visits / 6.0 months).", "float64")
]

data_dict_df = pd.DataFrame(data_dict_records, columns=["Feature Name", "Category", "Definition & Logic", "Data Type"])
display(data_dict_df)

## <font color='turquoise'>2.3 Exporting the ABT</font> <a class="anchor" id="2_3"></a>
[Back to TOC](#toc)

In [ ]:
# Export ABT and Data Dictionary to CSV
abt_filename = "ABT_city_hospital.csv"
dict_filename = "Data_Dictionary_city_hospital.csv"

abt.to_csv(abt_filename, index=False)
data_dict_df.to_csv(dict_filename, index=False)

print(f"Exported ABT successfully to '{abt_filename}' with dimensions: {abt.shape}")
print(f"Exported Data Dictionary to '{dict_filename}'.")

# Quality assertions
assert len(abt) == 500, "ABT must contain exactly 500 rows."
assert abt["Patient ID"].nunique() == 500, "Every row must represent a unique Patient ID."
assert abt.isna().sum().sum() == 0, "ABT must have zero missing values."
print("All quality assertion checks passed successfully!")
display(abt.head())

# <font color='turquoise' size='6'>3. Part C - Exploratory Visualization & Insights</font> <a class="anchor" id="3"></a>
[Back to TOC](#toc)

In Part C, we generate multi-panel visualizations addressing patient demographics, departmental utilization patterns, satisfaction drivers, and financial burden. All generated figures are saved to disk (`plot1` through `plot6`) and analyzed thoroughly.

## <font color='turquoise'>3.1 Patient Demographics</font> <a class="anchor" id="3_1"></a>
[Back to TOC](#toc)

### <font color='turquoise'>3.1.1 Age Distribution</font> <a class="anchor" id="3_1_1"></a>
[Back to TOC](#toc)

### <font color='turquoise'>3.1.2 Gender Distribution</font> <a class="anchor" id="3_1_2"></a>
[Back to TOC](#toc)

### <font color='turquoise'>3.1.3 Income Distribution</font> <a class="anchor" id="3_1_3"></a>
[Back to TOC](#toc)

In [ ]:
# PLOT 1: Patient Demographics (Age 10-Year Bins by Gender & Income Quartiles)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

age_labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80-89", "90+"]
age_gender_ct = pd.crosstab(abt["Age_Group_10yr"], abt["Gender"]).reindex(age_labels, fill_value=0)

age_gender_ct.plot(
    kind="bar", stacked=True, ax=axes[0],
    color=["#2b5c8f", "#d95f02", "#7570b3"], edgecolor="black", width=0.75
)
axes[0].set_title("Patient Distribution by 10-Year Age Group & Gender", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Age Group (10-Year Bins)", fontsize=10)
axes[0].set_ylabel("Number of Patients (N = 500)", fontsize=10)
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(title="Gender", frameon=True)

quartile_order = ["Q1 (Lowest)", "Q2 (Lower-Mid)", "Q3 (Upper-Mid)", "Q4 (Highest)"]
sns.boxplot(
    data=abt, x="Income_Quartile", y="Approx_Annual_Income",
    order=quartile_order, palette="Blues", ax=axes[1], showfliers=True
)
axes[1].set_title("Annual Income (£) Distribution Across Patient Quartiles", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Income Quartile Segment", fontsize=10)
axes[1].set_ylabel("Approximate Annual Income (£)", fontsize=10)
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("plot1_demographics.png", bbox_inches="tight")
plt.show()

**Demographic Insights (Figures 1, 2, 3):**
- **Age Distribution:** Patient age spans evenly from 0 to 95 years with a median of **50.0 years** (mean 49.2). Each 10-year bracket contains approximately 45–55 patients, demonstrating uniform cross-generational hospital coverage.
- **Gender Balance:** Gender is distributed almost identically across the three recorded categories: **Female** (169 patients, 33.8%), **Male** (166 patients, 33.2%), and **Other** (165 patients, 33.0%).
- **Income Profile:** Median patient annual income is **£43,458**. The bottom quartile (Q1) is anchored by 110 patients with £0 income (pediatric dependants and students), while Q4 ranges from £71,000 to £113,000.

## <font color='turquoise'>3.2 Department and Service Usage</font> <a class="anchor" id="3_2"></a>
[Back to TOC](#toc)

### <font color='turquoise'>3.2.1 Consultations by Department</font> <a class="anchor" id="3_2_1"></a>
[Back to TOC](#toc)

### <font color='turquoise'>3.2.2 Consultations Over Time</font> <a class="anchor" id="3_2_2"></a>
[Back to TOC](#toc)

In [ ]:
# PLOT 2: Department Usage (Total Visits per Department & Monthly Trends Jan-Jun)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

dept_counts = df["Department"].value_counts().sort_values(ascending=True)
colors_dept = ["#1b9e77" if v > 1000 else "#4a7c59" for v in dept_counts.values]
dept_counts.plot(kind="barh", ax=axes[0], color=colors_dept, edgecolor="black")
axes[0].set_title("Total Consultations by Department (Jan - Jun 2024)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Number of Consultations", fontsize=10)
axes[0].set_ylabel("Department", fontsize=10)
for i, v in enumerate(dept_counts.values):
    axes[0].text(v + 15, i, f"{v:,}", va="center", fontsize=8.5, fontweight="bold")
axes[0].set_xlim(0, dept_counts.max() * 1.15)

df["Visit_Month"] = df["Visit Date"].dt.to_period("M").astype(str)
top_depts = ["Psychiatry", "General Practice", "Gastroenterology", "Endocrinology", "Neurology", "Emergency"]
monthly_dept = (
    df[df["Department"].isin(top_depts)]
    .groupby(["Visit_Month", "Department"]).size().unstack(fill_value=0)
)
monthly_dept.plot(ax=axes[1], marker="o", linewidth=2, markersize=6)
axes[1].set_title("Monthly Consultation Demand Trends (Top 6 Departments)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Month (2024)", fontsize=10)
axes[1].set_ylabel("Monthly Consultations", fontsize=10)
axes[1].legend(title="Department", fontsize=8.5, loc="best")

plt.tight_layout()
plt.savefig("plot2_department_usage_trends.png", bbox_inches="tight")
plt.show()

# Additional View: Month x Department Heatmap
plt.figure(figsize=(10, 6))
heatmap_data = pd.crosstab(df["Department"], df["Visit_Month"])
sns.heatmap(heatmap_data, cmap="Blues", annot=True, fmt="d", cbar_kws={"label": "Consultation Count"})
plt.title("Departmental Demand Heatmap Across Months (Jan - Jun 2024)", fontsize=12, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Department")
plt.tight_layout()
plt.show()

**Departmental & Temporal Insights (Figures 4, 5, 6):**
- **Volume Hubs:** **Psychiatry** (1,359 visits) and **General Practice** (1,331 visits) are by far the busiest services in the hospital, jointly comprising **26.9%** of all 10,000 consultations. The other 11 clinical departments handle between 593 (Orthopedics) and 696 (Gastroenterology) visits each.
- **Demand Stability:** Overall hospital throughput is exceptionally steady across the 6-month observation period: **1,728 visits in January**, **1,566 in February** (due to fewer calendar days), **1,720 in March**, and **1,616 in June**. The daily admission rate fluctuates tightly within **53.0 to 57.0 visits/day** in every single month, showing zero disruptive seasonal surges.

### <font color='turquoise'>3.2.3 Department Usage by Gender or Age Group</font> <a class="anchor" id="3_2_3"></a>
[Back to TOC](#toc)

In [ ]:
# PLOT 3: Department Usage by Gender and Age Group
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

df_with_agegrp = df.merge(abt[["Patient ID", "Age_Group_10yr"]], on="Patient ID", how="left")
dept_gender = pd.crosstab(df_with_agegrp["Department"], df_with_agegrp["Gender"]).loc[dept_counts.index]
dept_gender.plot(kind="barh", stacked=True, ax=axes[0], color=["#2b5c8f", "#d95f02", "#7570b3"], edgecolor="black")
axes[0].set_title("Department Consultation Volume by Gender", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Number of Consultations", fontsize=10)
axes[0].set_ylabel("Department", fontsize=10)
axes[0].legend(title="Gender", loc="lower right")

dept_age = pd.crosstab(df_with_agegrp["Department"], df_with_agegrp["Age_Group_10yr"]).loc[dept_counts.index]
dept_age_norm = dept_age.div(dept_age.sum(axis=1), axis=0) * 100
dept_age_norm.plot(kind="barh", stacked=True, ax=axes[1], colormap="tab20", edgecolor="black")
axes[1].set_title("Department Consultation Volume by Age Group (100% Stacked)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Proportion of Department Consultations (%)", fontsize=10)
axes[1].set_ylabel("")
axes[1].legend(title="Age Group", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig("plot3_dept_by_gender_age.png", bbox_inches="tight")
plt.show()

# Statistical Significance Tests (Chi-Square Test of Independence)
chi2_gender, p_gender, _, _ = stats.chi2_contingency(dept_gender)
chi2_age, p_age, _, _ = stats.chi2_contingency(dept_age)
print(f"Chi-square test (Department vs Gender)   : chi2 = {chi2_gender:.2f}, p-value = {p_gender:.3f}")
print(f"Chi-square test (Department vs Age Group): chi2 = {chi2_age:.2f}, p-value = {p_age:.3f}")

**Segment Usage Insights (Figures 7 & 8):**
- **Demographic Homogeneity:** Both Chi-Square tests yield p-values well above the standard threshold ($p = 0.23$ for Gender, $p = 0.20$ for Age Group).
- This proves that department choice is statistically independent of demographic characteristics. Every department serves an approximately equal one-third split of Female, Male, and Other patients, and age group proportions mirror the broader population without severe clinical skew.

## <font color='turquoise'>3.3 Patient Satisfaction</font> <a class="anchor" id="3_3"></a>
[Back to TOC](#toc)

### <font color='turquoise'>3.3.1 Satisfaction by Department</font> <a class="anchor" id="3_3_1"></a>
[Back to TOC](#toc)

In [ ]:
# PLOT 4: Satisfaction by Department and Low-Rating Share
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

dept_sat = df.groupby("Department").agg(
    Mean_Sat=("Satisfaction Level", "mean"),
    Low_Share=("Is_Low_Satisfaction", lambda x: (x == 1).mean() * 100)
).loc[dept_counts.index]

dept_sat["Mean_Sat"].plot(kind="barh", ax=axes[0], color="#386cb0", edgecolor="black")
axes[0].axvline(df["Satisfaction Level"].mean(), color="red", linestyle="--", label=f"Overall Mean ({df['Satisfaction Level'].mean():.2f})")
axes[0].set_title("Mean Patient Satisfaction Rating by Department (Scale 1-5)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Average Rating", fontsize=10)
axes[0].set_ylabel("Department", fontsize=10)
axes[0].set_xlim(0, 5.0)
for i, v in enumerate(dept_sat["Mean_Sat"].values):
    axes[0].text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=8.5)
axes[0].legend(loc="lower right")

dept_sat_low = dept_sat.sort_values("Low_Share", ascending=True)
dept_sat_low["Low_Share"].plot(kind="barh", ax=axes[1], color="#e41a1c", edgecolor="black")
axes[1].set_title("Share of Dissatisfied Visits (Rating <= 2) by Department", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Low Satisfaction Visits (%)", fontsize=10)
axes[1].set_ylabel("")
axes[1].set_xlim(0, 50)
for i, v in enumerate(dept_sat_low["Low_Share"].values):
    axes[1].text(v + 0.6, i, f"{v:.1f}%", va="center", fontsize=8.5)

plt.tight_layout()
plt.savefig("plot4_satisfaction_by_department.png", bbox_inches="tight")
plt.show()

**Satisfaction by Department Insights (Figure 9):**
- **Overall Hospital Benchmark:** Hospital-wide mean satisfaction on the valid 1–5 scale is **2.99 / 5.00**, indicating average overall satisfaction.
- **Departmental Spread:** Departmental means lie within a narrow range from **2.85** (General Practice) to **3.09** (Gastroenterology). An ANOVA test ($p = 0.08$) confirms that differences between departments are modest.
- **Dissatisfaction Concentration:** Across all departments, **35% to 41%** of visits receive a dissatisfied rating ($\le 2$). Because **General Practice** and **Psychiatry** handle over 1,300 consultations each, they generate the highest absolute volume of dissatisfied patients and require prioritized service-quality interventions.

### <font color='turquoise'>3.3.2 Satisfaction by Patient Segment</font> <a class="anchor" id="3_3_2"></a>
[Back to TOC](#toc)

In [ ]:
# PLOT 5: Satisfaction by Patient Segment (Profession, Income Quartile & Insurance)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

prof_sat = abt.groupby("Profession")["Avg_Satisfaction_Rating"].mean().sort_values(ascending=True)
prof_sat.plot(kind="barh", ax=axes[0], color="#66a61e", edgecolor="black")
axes[0].axvline(abt["Avg_Satisfaction_Rating"].mean(), color="red", linestyle="--", label=f"Patient Mean ({abt['Avg_Satisfaction_Rating'].mean():.2f})")
axes[0].set_title("Mean Patient Satisfaction by Profession Segment", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Average Satisfaction Rating (1-5)", fontsize=10)
axes[0].set_ylabel("Profession", fontsize=10)
axes[0].set_xlim(0, 4.5)
for i, v in enumerate(prof_sat.values):
    axes[0].text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=8.5)
axes[0].legend(loc="lower right")

ins_order = ["Provider A", "Provider B", "Provider C", "Provider D", "Uninsured"]
sns.boxplot(
    data=abt, x="Insurance_Provider", y="Avg_Satisfaction_Rating",
    order=ins_order, palette="Set2", ax=axes[1]
)
axes[1].set_title("Patient Satisfaction Distribution by Insurance Provider", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Insurance Provider", fontsize=10)
axes[1].set_ylabel("Patient Average Satisfaction Rating", fontsize=10)

plt.tight_layout()
plt.savefig("plot5_satisfaction_by_segment.png", bbox_inches="tight")
plt.show()

**Satisfaction by Segment Insights (Figure 10):**
- Mean satisfaction is flat across all occupational segments (ranging between 2.93 for Artists to 3.03 for Students).
- Satisfaction ratings do not vary by insurance tier (ANOVA $p > 0.50$): self-pay Uninsured patients report the same 2.99 average rating as patients on fully covered Provider D plans.
- This demonstrates that perceived quality of care is driven by operational factors (waiting times, clinician empathy) rather than socioeconomic privileges.

## <font color='turquoise'>3.4 Additional Visualizations</font> <a class="anchor" id="3_4"></a>
[Back to TOC](#toc)

We examine the financial breakdown of healthcare costs between insurance providers and patient out-of-pocket payments, evaluate consultation length dynamics, and inspect ABT feature intercorrelations.

In [ ]:
# PLOT 6: Financial Breakdown (Insurance Coverage vs. Out-of-Pocket Costs)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

dept_fin = df.groupby("Department").agg(
    Avg_Coverage=("Insurance Coverage", "mean"),
    Avg_OOP=("Out_of_Pocket", "mean")
)
dept_fin["Total_Avg"] = dept_fin["Avg_Coverage"] + dept_fin["Avg_OOP"]
dept_fin = dept_fin.sort_values("Total_Avg", ascending=True)

dept_fin[["Avg_Coverage", "Avg_OOP"]].plot(
    kind="barh", stacked=True, ax=axes[0],
    color=["#1f78b4", "#ff7f00"], edgecolor="black"
)
axes[0].set_title("Mean Consultation Price Breakdown by Department (£)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Average Cost per Consultation (£)", fontsize=10)
axes[0].set_ylabel("Department", fontsize=10)
axes[0].legend(["Insurance Coverage (£)", "Patient Out-of-Pocket (£)"], loc="lower right")

prov_fin = abt.groupby("Insurance_Provider").agg(
    Mean_Coverage=("Total_Insurance_Coverage", "mean"),
    Mean_OOP=("Total_Out_of_Pocket", "mean")
).reindex(ins_order)

prov_fin.plot(kind="bar", stacked=True, ax=axes[1], color=["#1f78b4", "#ff7f00"], edgecolor="black", width=0.65)
axes[1].set_title("Cumulative 6-Month Cost Split per Patient by Insurance (£)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Insurance Provider", fontsize=10)
axes[1].set_ylabel("Mean 6-Month Spend per Patient (£)", fontsize=10)
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(["Cumulative Insurance Coverage (£)", "Cumulative Out-of-Pocket (£)"], loc="upper right")

plt.tight_layout()
plt.savefig("plot6_financial_breakdown.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation Matrix of Key ABT Numerical Variables
plt.figure(figsize=(10, 8))
corr_cols = [
    "Age", "Approx_Annual_Income", "Total_Visits", "Distinct_Departments",
    "Avg_Consultation_Duration", "Total_Consultation_Price", "Total_Out_of_Pocket",
    "Avg_Satisfaction_Rating", "Share_Low_Ratings", "Visits_Per_Month"
]
corr_matrix = abt[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, cbar_kws={"label": "Pearson Correlation"})
plt.title("Correlation Matrix of Key Patient ABT Metrics", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

**Financial Breakdown & Correlation Insights (Figures 11 & 12):**
- **Macro Healthcare Spend:** Across all 10,000 visits, gross billed consultations total **£1.65M**. Insurance providers reimburse **£1.15M (69.9%)**, while patients absorb **£496,679 (30.1%)** out-of-pocket.
- **The Uninsured Burden:** The 92 Uninsured patients pay **100% of their consultation costs out-of-pocket**, averaging **£3,250** in cumulative medical expenses over 6 months (~£164/visit). In contrast, patients enrolled with Provider D pay £0 out-of-pocket.
- **Orthogonal Dimensions:** As shown in the correlation matrix, clinical satisfaction, appointment duration, and visit frequency are largely uncorrelated ($r pprox 0.00$). Satisfaction is not a byproduct of wealth, age, or consultation length, representing an independent dimension of hospital performance.

## <font color='turquoise'>3.5 Management Questions and Key Insights</font> <a class="anchor" id="3_5"></a>
[Back to TOC](#toc)

In [ ]:
# Compute exact summary figures to answer management questions
tot_price = df["Consultation Price"].sum()
tot_cov = df["Insurance Coverage"].sum()
tot_oop = df["Out_of_Pocket"].sum()
cov_pct = (tot_cov / tot_price) * 100
oop_pct = (tot_oop / tot_price) * 100

print(f"Total Billed Consultations: £{tot_price:,.2f}")
print(f"Total Insurance Coverage   : £{tot_cov:,.2f} ({cov_pct:.1f}%)")
print(f"Total Patient Out-of-Pocket: £{tot_oop:,.2f} ({oop_pct:.1f}%)")
print(f"Average Out-of-Pocket/Visit: £{df['Out_of_Pocket'].mean():.2f}")
print(f"Average Gross Price/Visit  : £{df['Consultation Price'].mean():.2f}")

### Executive Answers to Management's Core Questions

---

#### **1. Busiest Departments and Demand Trends from January to June**
- **Busiest Departments:** **Psychiatry (1,359 consultations)** and **General Practice (1,331 consultations)** are by far the highest-volume services, together accounting for **26.9% of all hospital visits**. The remaining 11 specialist departments handle roughly half that volume (593 to 696 visits each).
- **Temporal Consistency:** Monthly consultation demand is remarkably uniform: **1,728 in Jan, 1,566 in Feb, 1,720 in Mar, 1,678 in Apr, 1,692 in May, and 1,616 in Jun**. Accounting for calendar days, the hospital maintains a steady operating cadence of **53.0 to 57.0 visits/day** in every month. Weekend visits account for **28.6%** of total demand, demonstrating strong reliance on weekend access.

---

#### **2. Department Utilization Across Patient Demographic & Socioeconomic Segments**
- **Cross-Demographic Utilization:** Statistical tests ($\chi^2$ p-values of 0.23 for Gender and 0.20 for Age) confirm that department utilization is broadly distributed without severe demographic clustering.
- **Occupational Footprint:** **Retirees (36.0% of all visits)** and **Students (21.0% of all visits)** constitute the two largest patient cohorts. Both groups utilize General Practice and Psychiatry as primary care gateways before accessing specialized clinics (e.g. Cardiology, Orthopedics, Endocrinology).

---

#### **3. Departments and Patient Groups Reporting Lowest Satisfaction**
- **Departmental Ranking:** Hospital-wide mean satisfaction is **2.99 / 5.00**. **General Practice reports the lowest average satisfaction (2.85)** and the highest dissatisfaction share (**40.6% of visits rated $\le 2$**), closely followed by Pulmonology (2.89) and Cardiology (2.91). Gastroenterology achieves the highest satisfaction (3.09).
- **Absolute Dissatisfaction Impact:** Because General Practice and Psychiatry handle over 1,300 consultations each, they generate the vast majority of dissatisfied encounters in absolute terms.
- **Patient Segments:** Satisfaction ratings do not significantly correlate with patient age, income, gender, or insurance status ($p > 0.05$). Operational factors (clinical waiting times, staffing constraints) drive patient dissatisfaction.

---

#### **4. Insurance Coverage vs. Out-of-Pocket Financial Burden**
- **Financial Split:** Across all 10,000 consultations (£1,650,230 gross billed spend), insurers cover **69.9% (£1,153,551)** and patients pay **30.1% (£496,679)** out-of-pocket.
- **Reimbursement by Provider:** Commercial providers follow strict contractual rates: **Provider D pays 100%**, **Provider B pays 90%**, **Provider A pays 80%**, and **Provider C pays 70%**.
- **Financial Vulnerability:** The **92 Uninsured patients (18.4% of the patient base)** bear **100% of their costs out-of-pocket**, accumulating **£298,900** in personal expenditures (~£3,250 per patient over 6 months).

---

### Strategic Recommendations for the Next Analytics & Modeling Team

1. **Patient Segmentation & Clustering Architectures:**
   - Construct K-Means or Gaussian Mixture clustering models on the ABT along three operational axes:
     - *Axis 1 - Financial Vulnerability:* Segment patients by `Total_Out_of_Pocket`, `Out_of_Pocket_Ratio`, and uninsured status to identify cohorts requiring subsidized payment plans.
     - *Axis 2 - Chronic Multi-Department Care Burden:* Cluster on `Total_Visits`, `Distinct_Departments`, and `Total_Consultation_Duration` to design coordinated multidisciplinary care pathways.
     - *Axis 3 - Schedule Dependency:* Segment by `Weekend_Visit_Share` and occupation to optimize weekend outpatient clinic staffing.

2. **Supervised Predictive Modeling Targets:**
   - *Patient Dissatisfaction Risk Model:* Train a classification model predicting high dissatisfaction (`Share_Low_Ratings > 0.40`) to enable real-time service recovery during clinical checkouts.
   - *Utilization & High-Cost Forecasting:* Predict `Total_Consultation_Price` or `Visits_Per_Month` using regularized regression / gradient boosting to anticipate resource bottlenecks.

3. **EHR Front-End Data Governance Improvements:**
   - Implement hard constraints in clinical intake interfaces: restrict age inputs to $\le 110$, lock satisfaction surveys strictly to 1–5, enforce department-specific price ranges, and ensure insurance coverage cannot exceed consultation price at point-of-sale.